# Baseline preparation and ledger rehearsal

Build five targets from saved fixtures, inspect abstentions and snapshot identity.
The optional Postgres section creates and removes a local disposable database.
Use the Baseline Python 3.12 kernel; offline cells take seconds, Postgres about 15 seconds.
All inputs are invented; these outputs are not official forecasts.


In [ ]:
import json
import tempfile
from pathlib import Path

import pandas as pd

from hoop.domains.nba.baselines.model import prepare
from hoop.platform.prepared import save_local
from hoop.platform.provenance import parse_time

ROOT = Path.cwd()
inputs = json.loads((ROOT / "tests/fixtures/nba/preparation.json").read_text())

## Run and inspect

The roster includes a player without history. Every roster member stays in the denominator. Minutes are averaged first; pooled rates use that same appearance window. Availability is descriptive in this baseline.


In [ ]:
result = prepare(inputs, generated_at=parse_time(inputs["fixture_generated_at"]))
display(pd.DataFrame(result["entries"])[["subject_id", "target", "value", "unit", "reason_code"]])
print("Prepared coverage:", result["coverage"])

In [ ]:
(ROOT / ".cache").mkdir(exist_ok=True)
with tempfile.TemporaryDirectory(dir=ROOT / ".cache") as folder:
    saved = save_local(result, Path(folder))
    replay = save_local(result, Path(folder))
    print("Snapshot:", saved["snapshot_id"], "| replay reused:", replay["reused"])

## Interpretation

Player 201 has 10 and 30 prior minutes, yielding 20 predicted minutes and 12 points. The rookie has five explicit abstentions. With complete common windows, this minutes/rate baseline equals each counting stat's recent mean; it is not evidence of predictive improvement. Saved inputs, code and configuration identify the prepared values. No database, live API, model activation or official publication runs here.


## Protected publication

SQL binds artifact bytes, both saved roster memberships and the active bundle. A second transaction observes the committed batch before the cutoff. Its receipt time is an upper bound on the first commit, not an exact commit timestamp.


In [ ]:
RUN_POSTGRES_REHEARSAL = False  # Opt in only after configuring local BASELINE_TEST_DSN.

In [ ]:
if RUN_POSTGRES_REHEARSAL:
    from scripts.rehearse_publication import run

    publication_report = run()
    display(pd.DataFrame([publication_report]))
else:
    publication_report = {"state": "not_run", "reason": "local Postgres opt-in required"}
    print(publication_report)

## Receipt interpretation

The optional run uses the same protected client as production and verifies an exact retry. Integration tests also cover rollback, lost insertion/receipt responses, late commits and invalidation. A batch without a timely receipt remains stored and unverified; recovery confirms that same identity. Production requires separate hosted/source/backup acceptance. Outcomes follow in OUTCOME-001.
